# XGBoost Encoding Benchmark – Test Evaluation

Loads **test data only** and evaluates all saved models from `models/`.

Run this notebook **after** `main_train_only.ipynb` has saved models.

| Step | Description |
|------|-------------|
| 0 | Set DEBUG flag |
| 1 | Load **test data only** |
| 2 | Auto-discover saved models in `models/` |
| 3 | Apply saved encoders to test data + generate predictions |
| 4 | Test-set lift charts for each model |
| 5 | Side-by-side comparison table |

---
## 0 · Setup & DEBUG flag

In [ ]:
import os, sys, glob, pickle, json
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "code"))

# ── Toggle this flag ─────────────────────────────────────────────────────────
DEBUG = 1   # 0 = full data  |  1 = 2K rows (fast smoke test)  |  2 = 10% sample (medium)
# ─────────────────────────────────────────────────────────────────────────────

MODELS_DIR  = os.path.join(PROJECT_ROOT, "models")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

_label = {0: 'FULL DATA', 1: '2K rows', 2: '10% sample'}.get(DEBUG, 'FULL DATA')
print(f"Project root : {PROJECT_ROOT}")
print(f"DEBUG        : {DEBUG}  ({_label})")

---
## 1 · Load test data only

Only test parquet is loaded here — no training data in memory.

| DEBUG | Data loaded |
|-------|-------------|
| 0 | Full test set |
| 1 | First 2K rows |
| 2 | Random 10% sample (matches DEBUG=2 training run) |

In [ ]:
from encoding_strategies import load_test_only, get_y

test_raw = load_test_only(debug=DEBUG)

y_test = get_y(test_raw)

print(f"y_test  : mean={y_test.mean():.4f}  std={y_test.std():.4f}")
print(f"Memory  : {test_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

---
## 2 · Discover saved models

Auto-scans `models/pp_bi_*/` for any trained experiments.

In [ ]:
# Auto-discover all saved model folders
model_dirs = sorted(glob.glob(os.path.join(MODELS_DIR, "pp_bi_*/")))

print(f"Found {len(model_dirs)} saved model(s):")
for d in model_dirs:
    name = os.path.basename(d.rstrip("/"))
    has_encoders = os.path.exists(os.path.join(d, "encoders.pkl"))
    print(f"  {name}  {'[encoders ✅]' if has_encoders else '[no encoders]'}")

---
## 3 · Helper: apply saved encoders to test data

Each encoding type has a different transform logic. The helper below
reconstructs `X_test` from the test raw data using the saved `encoders.pkl`.

In [ ]:
from encoding_strategies import (
    _get_numeric_features, _get_object_features,
    _transform_ohe, _apply_binary_map, _apply_h_map2to4, _is_0_5_col,
    EXCLUDE_ALWAYS
)


def apply_encoders_to_test(test_df: pd.DataFrame, encoders: dict, enc_type: str) -> pd.DataFrame:
    """Reconstruct X_test from test_df using saved encoders.

    Parameters
    ----------
    test_df   : raw test DataFrame
    encoders  : dict loaded from encoders.pkl
    enc_type  : one of 'type1_ordinal', 'type2_binary', 'type3_actuarial', 'type4_custom'
                (partial match is used — just needs to contain the type keyword)

    Returns
    -------
    X_test : encoded DataFrame
    """
    num_cols = _get_numeric_features(test_df)
    obj_cols = _get_object_features(test_df)

    # ── Type 1 Ordinal ────────────────────────────────────────────────────────
    if "type1" in enc_type:
        X_te = test_df[num_cols].astype("float32").copy()
        ohe_parts = []
        for col in obj_cols:
            if col in encoders:
                ohe_parts.append(_transform_ohe(col, test_df[col].astype(str), encoders[col]))
        if ohe_parts:
            X_te = pd.concat([X_te] + ohe_parts, axis=1)
        return X_te

    # ── Type 2 Binary ─────────────────────────────────────────────────────────
    elif "type2" in enc_type:
        low_set  = {0, 1, 2}
        high_set = {3, 4, 5}
        X_te = pd.DataFrame(index=test_df.index)
        for col in num_cols:
            if _is_0_5_col(test_df[col]):
                X_te[col] = _apply_binary_map(test_df[col], low_set, high_set)
            else:
                X_te[col] = test_df[col].astype("float32")
        ohe_parts = []
        for col in obj_cols:
            if col in encoders:
                ohe_parts.append(_transform_ohe(col, test_df[col].astype(str), encoders[col]))
        if ohe_parts:
            X_te = pd.concat([X_te] + ohe_parts, axis=1)
        return X_te

    # ── Type 3 Actuarial ──────────────────────────────────────────────────────
    elif "type3" in enc_type:
        strategy_map = encoders.get("__strategy_map__", {})
        enc_num_cols = encoders.get("__num_cols__", num_cols)
        enc_obj_cols = encoders.get("__obj_cols__", obj_cols)
        all_cols     = enc_num_cols + enc_obj_cols

        LOW_HI  = ({0,1,2}, {3,4,5})
        LO_HIGH = ({0,1,2,3}, {4,5})

        X_te_parts = []
        for col in all_cols:
            if col not in test_df.columns:
                continue
            strat = strategy_map.get(col, "not_specified")
            if strat == "drop":
                continue
            elif strat in ("ordered", "not_specified"):
                if col in enc_num_cols:
                    X_te_parts.append(test_df[[col]].astype("float32"))
                else:
                    cats = encoders[col][1]
                    te_s = pd.Series(pd.Categorical(test_df[col], categories=cats).codes,
                                     index=test_df.index, name=col, dtype="float32")
                    X_te_parts.append(te_s.to_frame())
            elif strat == "binary_low_hi":
                X_te_parts.append(_apply_binary_map(test_df[col], *LOW_HI).rename(col).to_frame())
            elif strat == "binary_lo_high":
                X_te_parts.append(_apply_binary_map(test_df[col], *LO_HIGH).rename(col).to_frame())
            elif strat in ("ohe", "group_ohe"):
                enc = encoders[col][1]
                X_te_parts.append(_transform_ohe(col, test_df[col].astype(str), enc))
            elif strat == "h_map2to4":
                X_te_parts.append(_apply_h_map2to4(test_df[col]).rename(col).to_frame())
            elif strat == "ohe_map2to4":
                enc = encoders[col][1]
                s_te = _apply_h_map2to4(test_df[col]).astype(str)
                X_te_parts.append(_transform_ohe(col, s_te, enc))
        return pd.concat(X_te_parts, axis=1)

    # ── Type 4 Custom (map 2->4, then binary {0,1,4}->0 | {3,5}->1) ─────────
    elif "type4" in enc_type:
        low_set  = {0, 1, 4}
        high_set = {3, 5}
        X_te = pd.DataFrame(index=test_df.index)
        for col in num_cols:
            if _is_0_5_col(test_df[col]):
                # Step 1: Remap 2->4
                remapped = _apply_h_map2to4(test_df[col])
                # Step 2: Apply binary grouping
                X_te[col] = _apply_binary_map(remapped, low_set, high_set)
            else:
                X_te[col] = test_df[col].astype("float32")
        ohe_parts = []
        for col in obj_cols:
            if col in encoders:
                ohe_parts.append(_transform_ohe(col, test_df[col].astype(str), encoders[col]))
        if ohe_parts:
            X_te = pd.concat([X_te] + ohe_parts, axis=1)
        return X_te

    else:
        raise ValueError(f"Unknown enc_type: '{enc_type}'. Expected type1/type2/type3/type4 in the name.")


print("✅ apply_encoders_to_test() helper loaded")

---
## 4 · Evaluate all saved models on test data

In [ ]:
from visualization import lift_chart

test_results = []

for model_dir in model_dirs:
    exp_name = os.path.basename(model_dir.rstrip("/")).replace("pp_bi_", "")
    model_path   = os.path.join(model_dir, "xgboost_model.pkl")
    feat_path    = os.path.join(model_dir, "feature_names.json")
    enc_path     = os.path.join(model_dir, "encoders.pkl")

    if not os.path.exists(model_path):
        print(f"  ⚠️ Skipping {exp_name} — no model file found")
        continue

    print(f"\n── Evaluating: {exp_name} ──────────────────────────────────────")

    # Load model
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    with open(feat_path) as f:
        feature_names = json.load(f)

    # Load encoders
    encoders = None
    if os.path.exists(enc_path):
        with open(enc_path, "rb") as f:
            encoders = pickle.load(f)

    # Reconstruct X_test
    if encoders is not None:
        X_test = apply_encoders_to_test(test_raw, encoders, enc_type=exp_name)
    else:
        # Fallback: select saved feature names directly (numeric only, no OHE)
        avail = [c for c in feature_names if c in test_raw.columns]
        X_test = test_raw[avail].astype("float32")

    # Align columns to exactly what the model expects
    missing = [c for c in feature_names if c not in X_test.columns]
    if missing:
        print(f"  ⚠️ {len(missing)} features missing from X_test — filling with 0")
        for c in missing:
            X_test[c] = 0.0
    X_test = X_test[feature_names].fillna(-1)

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae  = float(mean_absolute_error(y_test, y_pred))
    r2   = float(r2_score(y_test, y_pred))

    print(f"    TEST  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")

    # Lift chart
    _result_te = {
        "experiment_name": exp_name,
        "y_pred": y_pred,
        "y_true": y_test.values,
        "metrics": {"rmse": rmse, "mae": mae, "r2": r2},
        "feature_names": feature_names,
        "model": model,
        "xgb_params": {},
    }
    lift_chart(_result_te, test_df=test_raw, bins=10, print_table=False,
               title=f'TEST Lift — {exp_name} | RMSE={rmse:.2f}')

    # Save test predictions
    pred_df = pd.DataFrame({"y_true": y_test.values, "y_pred": y_pred,
                            "residual": y_test.values - y_pred})
    pred_df.to_csv(os.path.join(model_dir, "test_predictions.csv"), index=False)

    test_results.append({
        "experiment": exp_name,
        "test_rmse":  round(rmse, 6),
        "test_mae":   round(mae,  6),
        "test_r2":    round(r2,   6),
        "n_features": len(feature_names),
    })

print("\n✅ All models evaluated")

---
## 5 · Final comparison table

In [ ]:
if test_results:
    summary = pd.DataFrame(test_results).set_index("experiment").sort_values("test_rmse")
    winner  = summary["test_rmse"].idxmin()

    print("\n" + "=" * 65)
    print("  TEST-SET ENCODING BENCHMARK  (sorted by RMSE ↑ best)")
    print("=" * 65)
    print(summary.to_string())
    print(f"\n  🏆 Best strategy: {winner}  "
          f"(RMSE={summary.loc[winner,'test_rmse']:.4f})")
    print("=" * 65)

    # Save CSV
    out_path = os.path.join(RESULTS_DIR, "test_benchmark_summary.csv")
    summary.reset_index().to_csv(out_path, index=False)
    print(f"\n  📄 Summary saved: {out_path}")
else:
    print("No results to compare — check that models were trained and saved correctly.")